# Part 1: Data Preparation and Test Collection

**CS533: Information Retrieval Systems**

This notebook examines the 1M-News dataset and documents the
data preparation steps used in the information retrieval experiments.

The dataset is divided into two periods:
- **P1:** 2005–2014
- **P2:** 2015–2025

The notebook covers:
1. Dataset exploration and temporal anomalies.
2. Article cleaning and duplicate removal.
3. Text preprocessing and tokenization.
4. Period splitting and training corpus statistics.
5. Time-split test collection and query analysis.

The existing project scripts produce the experimental artifacts.
This notebook focuses on inspecting and verifying the results
without retraining models or rebuilding the retrieval indexes.

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Project and dataset locations
REPO_ROOT = Path(r"C:\Projects\IR\CS533_HW1")
DATA_ROOT = REPO_ROOT.parent / "data"
ARTICLES_PATH = DATA_ROOT / "articles_mapped.h5"

print("Project directory:", REPO_ROOT)
print("Dataset found:", ARTICLES_PATH.is_file())

Project directory: C:\Projects\IR\CS533_HW1
Dataset found: True


### 1.1 Dataset Exploration
We begin by examining the original 1M-News dataset before applying any cleaning rules. The dataset contains New York Times articles published between 2005 and 2025, assigned to nine broad categories.

We analyze the number of articles, their distribution across years and categories, and potential gaps in temporal coverage. These checks help us understand the dataset before dividing it into the two experimental periods.

In [4]:

# Load the original dataset without modifying the source file.
df = pd.read_hdf(ARTICLES_PATH, key="data")

# Extract publication year and assign experimental periods.
df["year"] = df["pub_date"].dt.year

df["period"] = "Outside"
df.loc[df["year"].between(2005, 2014), "period"] = "P1"
df.loc[df["year"].between(2015, 2025), "period"] = "P2"

print(f"Total articles: {len(df):,}")
print(f"Original data columns: {df.shape[1] - 2}")
print(f"Publication years: {df['year'].min()}–{df['year'].max()}")
print(f"Categories: {df['mapped_section'].nunique()}")
print(f"Articles outside P1/P2: {(df['period'] == 'Outside').sum():,}")

display(df[["pub_date", "mapped_section", "period"]].head())


Total articles: 1,007,004
Original data columns: 5
Publication years: 2005–2025
Categories: 9
Articles outside P1/P2: 0


,pub_date,mapped_section,period
_id,,,
nyt://article/0ab7e750-5a63-5ce3-8274-b61e5930ac05,2005-01-01 05:00:00+00:00,"Culture, Arts & Lifestyle",P1
nyt://article/0ccaa5bc-2e61-5965-937c-0e1a35be669b,2005-01-01 05:00:00+00:00,International News,P1
nyt://article/0d1d5b9a-9826-51cf-9f36-ff4437ad368a,2005-01-01 05:00:00+00:00,International News,P1
nyt://article/1413b21a-d4db-5588-9caf-9f93593643c2,2005-01-01 05:00:00+00:00,International News,P1
nyt://article/1e015c8d-9f7f-5f1c-a96f-43609868c2ae,2005-01-01 05:00:00+00:00,Politics & Government,P1


In [5]:

# Article counts for each year and category.
year_category = pd.crosstab(
    df["year"],
    df["mapped_section"]
)
year_category["Total"] = year_category.sum(axis=1)

print("Articles by year and category:")
display(year_category)

# Category distributions across both experimental periods.
period_category = pd.crosstab(
    df["mapped_section"],
    df["period"]
).reindex(columns=["P1", "P2"], fill_value=0)

period_category.loc["Total"] = period_category.sum()

print("Articles by category and period:")
display(period_category)


Articles by year and category:


mapped_section,Business & Economy,"Culture, Arts & Lifestyle",Health & Wellbeing,International News,Lifestyle & Leisure,Other,Politics & Government,Sports & Athletics,"Tech, Science & Education",Total
year,,,,,,,,,,
2005,15841,11800,1418,25981,4048,289,10949,7138,2029,79493
2006,26969,16321,1868,32215,4999,854,14688,11946,3602,113462
2007,18128,9577,996,15592,4071,1179,11612,9040,3811,74006
2008,13245,6510,1054,9707,3332,706,10292,7721,2880,55447
2009,1283,1609,270,1556,806,81,694,750,405,7454
2010,5333,6060,828,6440,1661,260,3122,3783,1501,28988
2011,6382,7285,924,7253,1894,1061,6553,5253,3594,40199
2012,11431,12955,1409,14210,3246,2068,12924,10485,7481,76209
2013,9539,10675,1176,12013,3096,1924,8697,6802,4736,58658


Articles by category and period:


period,P1,P2
mapped_section,,
Business & Economy,119367,52164
"Culture, Arts & Lifestyle",94516,87790
Health & Wellbeing,11069,19449
International News,137829,84113
Lifestyle & Leisure,31106,21071
Other,9966,8716
Politics & Government,89309,79553
Sports & Athletics,68483,29028
"Tech, Science & Education",34063,29412


In [10]:

# Examine the decline in Sports & Athletics.
sports_data = df.loc[
    df["mapped_section"] == "Sports & Athletics",
    ["year", "pub_date"]
].copy()

sports_data["month"] = sports_data["pub_date"].dt.month

sports_monthly = (
    sports_data.groupby(["year", "month"])
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=[2023, 2024, 2025],
        columns=range(1, 13),
        fill_value=0
    )
)

print("Monthly Sports & Athletics counts (2023–2025):")
display(sports_monthly)


Monthly Sports & Athletics counts (2023–2025):


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2023,169,144,185,188,175,162,173,127,97,14,8,7
2024,2,2,6,10,8,11,13,8,12,5,4,6
2025,8,11,6,3,1,0,0,0,0,0,0,0


### 1.1  Key Findings

The dataset contains **1,007,004 articles** across nine categories (2005–2025). Before cleaning, P1 contains **595,708 articles** and P2 contains **411,296 articles**.

- **Category distribution:** International News dominates P1 (137,829 articles), while Culture, Arts & Lifestyle leads P2 (87,790), indicating changes in category composition.
- **2009 anomaly:** Only 7,454 articles are present, all from November and December, indicating incomplete coverage.
- **2025 anomaly:** Articles appear only from January through May, with a sharp decline in May (693 articles), confirming partial-year coverage.
- **Sports & Athletics:** Monthly counts drop from 97 in September 2023 to 14 in October. Only 87 articles appear in 2024, suggesting a coverage or classification discontinuity.

**Conclusion:** Uneven temporal coverage and category distributions may influence comparisons between P1 and P2.

### 1.2  Data Cleaning

We clean the original dataset using three sequential rules:

1. Remove articles containing fewer than 10 whitespace-separated tokens.
2. Remove duplicate article text, keeping the first occurrence.
3. Remove duplicate URLs, keeping the first occurrence.

The order matters because each step operates on the articles retained by the previous step. Cleaning is performed in memory without modifying the original dataset or saved experimental artifacts.

In [12]:

# Step 1: Remove articles with fewer than 10 whitespace tokens.
short_mask = df["text"].fillna("").str.count(r"\S+") < 10
short_removed = int(short_mask.sum())
clean_df = df.loc[~short_mask].copy()

# Step 2: Remove duplicate text (keep first occurrence).
text_mask = clean_df["text"].duplicated(keep="first")
text_removed = int(text_mask.sum())
clean_df = clean_df.loc[~text_mask].copy()

# Step 3: Remove duplicate URLs (keep first occurrence).
url_mask = clean_df["web_url"].duplicated(keep="first")
url_removed = int(url_mask.sum())
clean_df = clean_df.loc[~url_mask].copy()

# Summarize the sequential cleaning results.
cleaning_summary = pd.DataFrame([
    ["Original", 0, len(df)],
    ["Fewer than 10 tokens", short_removed, len(df) - short_removed],
    ["Duplicate article text", text_removed,
     len(df) - short_removed - text_removed],
    ["Duplicate URL", url_removed, len(clean_df)]
], columns=["Step", "Removed", "Remaining"])

print("Cleaning summary:")
display(cleaning_summary)

print("\nCleaned articles by period:")
display(
    clean_df["period"].value_counts()
    .reindex(["P1", "P2"])
    .rename("Articles")
    .to_frame()
)

print("\nCleaned articles by category and period:")
display(pd.crosstab(clean_df["mapped_section"], clean_df["period"]))

# Verify accounting.
assert len(clean_df) == len(df) - (
    short_removed + text_removed + url_removed
)
assert clean_df["period"].isin(["P1", "P2"]).all()

print(f"\nFinal cleaned articles: {len(clean_df):,}")


Cleaning summary:


,Step,Removed,Remaining
0,Original,0,1007004
1,Fewer than 10 tokens,28799,978205
2,Duplicate article text,6914,971291
3,Duplicate URL,571,970720



Cleaned articles by period:


,Articles
period,
P1,563736
P2,406984



Cleaned articles by category and period:


period,P1,P2
mapped_section,,
Business & Economy,112719,51842
"Culture, Arts & Lifestyle",91093,86368
Health & Wellbeing,10670,19443
International News,126254,84085
Lifestyle & Leisure,29067,20747
Other,9894,6697
Politics & Government,84715,79464
Sports & Athletics,66950,29016
"Tech, Science & Education",32374,29322



Final cleaned articles: 970,720


#### 1.2  Key Findings

Sequential cleaning removed **36,284 articles (3.6%)**, leaving **970,720 articles**.

- **Short articles:** 28,799 removed (fewer than 10 tokens).
- **Duplicate text:** 6,914 removed.
- **Duplicate URLs:** 571 removed.

The cleaned dataset contains **563,736 articles in P1** and **406,984 in P2**. Most removals occurred in P1 (31,972 articles), indicating that cleaning affected the two periods unevenly.

The cleaning rules were applied in the required order, keeping the first occurrence of each duplicate.

### 1.3 Text Preprocessing

We use a consistent tokenization pipeline for embedding training, indexing, and queries. The pipeline handles capitalization, Unicode quotes, punctuation, acronyms, possessives, and numbers. Stop words are retained for embedding training.

This section verifies the preprocessing rules, demonstrates two before/after examples, and reports total and distinct token counts for P1 and P2.

In [14]:

import sys
import inspect

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import preprocess_text

# Test preprocessing decisions using controlled examples.
examples = [
    "The N.B.A. team's 2025 pre-season.",
    "U.S. researchers discussed AI-powered tools and café prices in 2024."
]

for i, example in enumerate(examples, start=1):
    print(f"Example {i}")
    print("Before:", example)
    print("After: ", preprocess_text(example))
    print()

# Test the same tokenizer on an actual cleaned article excerpt.
article_excerpt = str(clean_df["text"].iloc[0])[:250]

print("Real article excerpt:")
print("Before:", article_excerpt)
print("After: ", preprocess_text(article_excerpt))
print()

# Inspect the function actually used by the project.
print("Tokenizer implementation:")
print(inspect.getsource(preprocess_text))


Example 1
Before: The N.B.A. team's 2025 pre-season.
After:  ['the', 'nba', 'team', '2025', 'pre', 'season']

Example 2
Before: U.S. researchers discussed AI-powered tools and café prices in 2024.
After:  ['us', 'researchers', 'discussed', 'ai', 'powered', 'tools', 'and', 'café', 'prices', 'in', '2024']

Real article excerpt:
Before: Techniques That Might Smile Upon Mona Lisa Article on research conducted by Smithsonian Institution engineer and painting conservator Marion Mecklenburg, who is seeking better ways to preserve priceless art works, including those like Mona Lisa that 
After:  ['techniques', 'that', 'might', 'smile', 'upon', 'mona', 'lisa', 'article', 'on', 'research', 'conducted', 'by', 'smithsonian', 'institution', 'engineer', 'and', 'painting', 'conservator', 'marion', 'mecklenburg', 'who', 'is', 'seeking', 'better', 'ways', 'to', 'preserve', 'priceless', 'art', 'works', 'including', 'those', 'like', 'mona', 'lisa', 'that']

Tokenizer implementation:
def preprocess_text(t

In [15]:

# Compare NYT-style acronyms with query-style spelling.
acronym_examples = [
    ("N.B.A. playoffs", "nba playoffs"),
    ("F.B.I. investigation", "fbi investigation"),
    ("E.P.A. regulations", "epa regulations"),
    ("U.S. economy", "us economy"),
]

results = []

for article_text, query_text in acronym_examples:
    article_tokens = preprocess_text(article_text)
    query_tokens = preprocess_text(query_text)

    results.append({
        "NYT text": article_text,
        "Query": query_text,
        "Article tokens": article_tokens,
        "Query tokens": query_tokens,
        "Match": article_tokens == query_tokens
    })

acronym_check = pd.DataFrame(results)
display(acronym_check)

assert acronym_check["Match"].all()
print("All acronym examples match.")


,NYT text,Query,Article tokens,Query tokens,Match
0,N.B.A. playoffs,nba playoffs,"[nba, playoffs]","[nba, playoffs]",True
1,F.B.I. investigation,fbi investigation,"[fbi, investigation]","[fbi, investigation]",True
2,E.P.A. regulations,epa regulations,"[epa, regulations]","[epa, regulations]",True
3,U.S. economy,us economy,"[us, economy]","[us, economy]",True


All acronym examples match.


In [16]:

import json

# Locate existing corpus files and saved results.
corpus_dir = DATA_ROOT / "corpora"
results_path = DATA_ROOT / "part1_review" / "part1_results.json"
count_script = REPO_ROOT / "src" / "token_counts.py"

print("Token-counting script exists:", count_script.is_file())
print("Saved results exist:", results_path.is_file())

print("\nSaved corpus files:")
if corpus_dir.is_dir():
    for path in sorted(corpus_dir.iterdir()):
        if path.is_file():
            print(f"{path.name}: {path.stat().st_size:,} bytes")
else:
    print("Corpus directory not found.")

# Inspect saved statistics without regenerating corpora.
if results_path.is_file():
    results = json.loads(results_path.read_text(encoding="utf-8"))
    print("\nSaved results sections:")
    print(list(results) if isinstance(results, dict) else type(results).__name__)

    expected = {47_477_057, 30_206_984, 231_482, 181_582}

    def find_counts(obj, location="results"):
        if isinstance(obj, dict):
            for key, value in obj.items():
                find_counts(value, f"{location}.{key}")
        elif isinstance(obj, list):
            for i, value in enumerate(obj):
                find_counts(value, f"{location}[{i}]")
        elif isinstance(obj, int) and not isinstance(obj, bool):
            if obj in expected:
                print(f"{location}: {obj:,}")

    print("\nMatching recorded token statistics:")
    find_counts(results)


Token-counting script exists: True
Saved results exist: True

Saved corpus files:
corpus_P1.txt: 277,388,445 bytes
corpus_P2.txt: 176,857,463 bytes

Saved results sections:
['passed', 'elapsed_seconds', 'source_paths', 'provenance', 'raw', 'cleaning', 'cleaned_count', 'training_article_counts', 'corpora', 'validations', 'official_reporting', 'processed_overlap', 'preprocessing_examples']

Matching recorded token statistics:
results.corpora.P1.total_tokens: 47,477,057
results.corpora.P1.distinct_tokens: 231,482
results.corpora.P2.total_tokens: 30,206,984
results.corpora.P2.distinct_tokens: 181,582


#### 1.3 Key Findings

**Preprocessing decisions:**
- **Lowercase:** Standardizes capitalization for consistent matching.
- **Unicode quotes:** Normalizes apostrophes before tokenization.
- **Regex tokenization:** Preserves Unicode letters and digits while separating punctuation and hyphens.
- **Acronyms:** Converts dotted forms such as `N.B.A.` to `nba` to match query spelling.
- **Possessives:** Removes endings such as `'s`; no stemming or lemmatization is applied.
- **Numbers:** Retains digits, although punctuation within numbers is split.
- **Stop words:** Retained for Word2Vec to preserve context and also retained for lexical retrieval.

**Consistency:** The same `preprocess_text` function is used for embedding training, document indexing, and query processing. All four tested acronym pairs match after preprocessing.

**Limitations:** `U.S.` and `us` become identical tokens. URL-derived known-item queries can also produce possessive mismatches, such as `comcasts` versus `comcast`.

**Corpus statistics:**

| Period | Total tokens | Distinct tokens |
|---|---:|---:|
| P1 | 47,477,057 | 231,482 |
| P2 | 30,206,984 | 181,582 |

The counts agree with the saved corpus-generation and training reports.

### 1.4 Training Corpora

The cleaned articles are divided into two non-overlapping periods: P1 (2005–2014) and P2 (2015–2025). Each corpus contains one preprocessed article per line, using all cleaned articles without downsampling.

This section reports the article and token counts from the saved training corpora.

In [18]:

# Report existing corpus statistics without regenerating files.
corpus_stats = []

for period in ["P1", "P2"]:
    corpus_path = corpus_dir / f"corpus_{period}.txt"
    saved = results["corpora"][period]

    corpus_stats.append({
        "Period": period,
        "Articles": int((clean_df["period"] == period).sum()),
        "Total tokens": saved["total_tokens"],
        "Distinct tokens": saved["distinct_tokens"],
        "File exists": corpus_path.is_file()
    })

display(pd.DataFrame(corpus_stats))

# Show recorded training article counts for comparison.
print("Saved training article counts:")
print(results["training_article_counts"])


,Period,Articles,Total tokens,Distinct tokens,File exists
0,P1,563736,47477057,231482,True
1,P2,406984,30206984,181582,True


Saved training article counts:
{'P1': 563736, 'P2': 406984}


### 1.5 Time-Split Test Collection

The test collection contains articles from two periods, C1 (2005–2014) and C2 (2015–2025), organized into 32 topics. Each period includes 1,000 known-item queries, while the same 32 topic queries are evaluated on both collections.

This section reports collection sizes, topic distributions, query-length statistics, example queries, query-term overlap, and validation against the official files.

In [20]:

# Load the official test collection files.
OFFICIAL_DIR = DATA_ROOT / "nyt-tc-2026"

collection_p1 = pd.read_csv(
    OFFICIAL_DIR / "collection_P1.tsv", sep="\t"
)
collection_p2 = pd.read_csv(
    OFFICIAL_DIR / "collection_P2.tsv", sep="\t"
)
queries_p1 = pd.read_csv(
    OFFICIAL_DIR / "queries_ki_P1.tsv", sep="\t"
)
queries_p2 = pd.read_csv(
    OFFICIAL_DIR / "queries_ki_P2.tsv", sep="\t"
)
topics = pd.read_csv(
    OFFICIAL_DIR / "topics.tsv", sep="\t"
)

# Report collection and query sizes.
summary = pd.DataFrame({
    "P1": [len(collection_p1), len(queries_p1)],
    "P2": [len(collection_p2), len(queries_p2)]
}, index=["Collection documents", "Known-item queries"])

print("Collection summary:")
display(summary)
print(f"Topic queries: {len(topics)}")

# Report the distribution of all 32 topics.
topic_distribution = pd.DataFrame({
    "C1": collection_p1["topic"].value_counts(),
    "C2": collection_p2["topic"].value_counts()
}).fillna(0).astype(int).sort_index()

print("\nTopic distribution:")
display(topic_distribution)

# Verify the totals.
assert len(collection_p1) == 173_603
assert len(collection_p2) == 210_664
assert len(queries_p1) == len(queries_p2) == 1_000
assert len(topics) == 32
assert topic_distribution["C1"].sum() == len(collection_p1)
assert topic_distribution["C2"].sum() == len(collection_p2)

print("\nCollection counts verified.")


Collection summary:


,P1,P2
Collection documents,173603,210664
Known-item queries,1000,1000


Topic queries: 32

Topic distribution:


,C1,C2
topic,,
africa,3684,2920
art,3996,6107
asia,12242,13380
baseball,12414,5047
basketball,5721,2725
books,3060,5998
dance,1255,2124
economy,1170,3123
europe,11193,20357



Collection counts verified.


In [23]:

# Query lengths measured in whitespace-separated tokens.
query_results = []

for period, queries in [("KI1", queries_p1), ("KI2", queries_p2)]:
    lengths = queries["query"].str.split().str.len()

    query_results.append({
        "Query set": period,
        "Queries": len(queries),
        "Mean": round(lengths.mean(), 2),
        "Median": lengths.median(),
        "Min": lengths.min(),
        "Max": lengths.max(),
        "Std": round(lengths.std(), 2)
    })

print("Known-item query-length statistics:")
display(pd.DataFrame(query_results))


# Display five complete queries per period.
with pd.option_context("display.max_colwidth", None):
    for period, queries in [("KI1", queries_p1), ("KI2", queries_p2)]:
        print(f"\n{period}: Five example queries")
        display(queries[["qid", "query", "docno"]].head(5))



Known-item query-length statistics:


,Query set,Queries,Mean,Median,Min,Max,Std
0,KI1,1000,7.79,8.0,3,16,2.05
1,KI2,1000,5.40,5.0,3,20,2.54



KI1: Five example queries


,qid,query,docno
0,KI1-0001,injuries to key players loom over spurs thunder series,C1-160665
1,KI1-0002,ties cut with iran institute over holocaust,C1-050517
2,KI1-0003,athletics edge yankees as andy pettitte struggles again,C1-140506
3,KI1-0004,internet customers and advertising sales bolster comcasts profits,C1-112385
4,KI1-0005,laura osnes star of rodgers hammersteins cinderella,C1-136714



KI2: Five example queries


,qid,query,docno
0,KI2-0001,brussels suspects photo,C2-026707
1,KI2-0002,israel gaza hostages dead,C2-186795
2,KI2-0003,it took years to realize theyve always been in the same room,C2-083702
3,KI2-0004,ennio morricone john zorn,C2-120602
4,KI2-0005,anthropic ai claude chatbot,C2-177076


In [24]:

def calculate_overlap(queries, collection, cleaned_articles):
    matched = (
        queries.merge(
            collection[["docno", "web_url"]],
            on="docno",
            validate="one_to_one"
        )
        .merge(
            cleaned_articles[["web_url", "text"]],
            on="web_url",
            validate="one_to_one"
        )
    )

    def overlap(row):
        query_tokens = preprocess_text(row["query"])
        if not query_tokens:
            return float("nan")

        article_tokens = set(preprocess_text(row["text"]))
        return sum(
            token in article_tokens for token in query_tokens
        ) / len(query_tokens)

    matched["overlap"] = matched.apply(overlap, axis=1)
    return matched


matched_p1 = calculate_overlap(queries_p1, collection_p1, clean_df)
matched_p2 = calculate_overlap(queries_p2, collection_p2, clean_df)

assert len(matched_p1) == len(queries_p1) == 1000
assert len(matched_p2) == len(queries_p2) == 1000

overlap_summary = pd.DataFrame({
    "Query set": ["KI1", "KI2"],
    "Matched queries": [len(matched_p1), len(matched_p2)],
    "Mean overlap (%)": [
        round(matched_p1["overlap"].mean() * 100, 2),
        round(matched_p2["overlap"].mean() * 100, 2)
    ]
})

print("Query-term overlap:")
display(overlap_summary)


Query-term overlap:


,Query set,Matched queries,Mean overlap (%)
0,KI1,1000,86.51
1,KI2,1000,83.50


In [29]:

# Summarize all official reconstruction comparisons.
validation_rows = []

for category, checks in results["validations"].items():
    for i, check in enumerate(checks, start=1):
        validation_rows.append({
            "Component": category,
            "Collection": check.get("collection", f"Check {i}"),
            "Differences": check.get("total_differences")
        })

validation_summary = pd.DataFrame(validation_rows)
display(validation_summary)

# Confirm every comparison reports zero differences.
if (
    validation_summary["Differences"].notna().all()
    and validation_summary["Differences"].eq(0).all()
):
    print("PASS: All 9 comparisons have zero differences.")
else:
    print("WARNING: Some comparisons have differences or missing results.")


,Component,Collection,Differences
0,collections,C1,0
1,collections,C2,0
2,known_item_queries,C1,0
3,known_item_queries,C2,0
4,known_item_qrels,C1,0
5,known_item_qrels,C2,0
6,topic_qrels,C1,0
7,topic_qrels,C2,0
8,topic_queries,Check 1,0


PASS: All 9 comparisons have zero differences.


#### 1.5 Key Findings

**Collection sizes:** C1 contains 173,603 documents, while C2 contains 210,664. Both collections cover 32 topics and include 1,000 known-item queries per period.

**Topic distribution:** Middle East is the largest topic in C1 (13,301 documents), while Politics dominates C2 (38,673). Topic distributions vary substantially between periods.

**Known-item query statistics:**

| Metric | KI1 | KI2 |
|---|---:|---:|
| Mean query length | 7.79 | 5.40 |
| Median length | 8 | 5 |
| Length range | 3–16 | 3–20 |
| Mean query-term overlap | 86.51% | 83.50% |

KI1 queries are longer and have a 3.01 percentage-point higher average overlap with their target articles. Differences in query wording and document vocabulary, including possessive mismatches, may contribute to this gap.

**Official validation:** All nine comparisons of reconstructed collections, queries, and relevance judgments report **zero differences** from the official files.